# Django, Advanced Validation

## Introduction to Advanced Validation
---

In this lesson, you'll learn how to **implement custom validation logic** in Django forms.

Beyond basic field validators, you'll create custom validation methods, cross-field validation, and reusable validators.

1. [Validation Flow](#Validation-Flow),
    - [How Django Validates Forms](#How-Django-Validates-Forms),
    - [Where Errors Are Stored](#Where-Errors-Are-Stored),
2. [Field-Level Validation](#Field-Level-Validation),
    - [clean_fieldname() Methods](#clean_fieldname()-Methods),
    - [Accessing Other Field Values](#Accessing-Other-Field-Values),
3. [Form-Level Validation](#Form-Level-Validation),
    - [The clean() Method](#The-clean()-Method),
    - [Cross-Field Validation](#Cross-Field-Validation),
4. [Custom Validators](#Custom-Validators),
    - [Creating Reusable Validators](#Creating-Reusable-Validators),
    - [Validators with Parameters](#Validators-with-Parameters),
5. [Error Handling and Display](#Error-Handling-and-Display),
    - [Adding Custom Errors](#Adding-Custom-Errors),
    - [Non-Field Errors](#Non-Field-Errors),
    - [Displaying Errors in Templates](#Displaying-Errors-in-Templates),
6. [🧠 EXERCISE 🧠, Password Validation Form](#🧠-EXERCISE-🧠,-Password-Validation-Form),
7. [🧠 EXERCISE 🧠, Date Range Validation](#🧠-EXERCISE-🧠,-Date-Range-Validation).

<br>

## Validation Flow

---

### How Django Validates Forms

---

Why does the validation process matter?

Validation in Django isn't just one big 'check.' It’s a sophisticated multi-stage pipeline.

If you don't understand the sequence, you'll end up fighting the framework—trying to access data that doesn't exist yet or wondering why your custom logic is being skipped.

When you call `form.is_valid()`, Django runs validation in a specific order:

<br>

```
is_valid()
    │
    ▼
┌─────────────────────────────────────────────────┐
│  1. to_python() - Convert input to Python type  │
└─────────────────────────────────────────────────┘
    │
    ▼
┌─────────────────────────────────────────────────┐
│  2. validate() - Run built-in field validation  │
└─────────────────────────────────────────────────┘
    │
    ▼
┌─────────────────────────────────────────────────┐
│  3. run_validators() - Run custom validators    │
└─────────────────────────────────────────────────┘
    │
    ▼
┌─────────────────────────────────────────────────┐
│  4. clean_<fieldname>() - Field-specific logic  │
└─────────────────────────────────────────────────┘
    │
    ▼ (after all fields)
┌─────────────────────────────────────────────────┐
│  5. clean() - Cross-field validation            │
└─────────────────────────────────────────────────┘
    │
    ▼
  Returns True/False
```

<br>

**Key insight:** Each step only runs if the previous step passed. If `to_python()` fails for a field, Django won't run `clean_<fieldname>()` for that field.

<br>

### Where Errors Are Stored

---

After validation, errors are stored in the form's `errors` attribute:

<br>

| Attribute/method | Description | Access |
|-----------|-------------|--------|
| `form.errors` | Dictionary of all errors | `{'email': ['Enter a valid email']}` |
| `form.errors['fieldname']` | Errors for specific field | `['Error 1', 'Error 2']` |
| `form.non_field_errors()` | Errors not tied to a field | Cross-field validation errors |

In [ ]:
# Inspecting form errors — using CommentCreateForm from our project

from django import forms


class CommentCreateForm(forms.Form):
    content = forms.CharField(min_length=3, widget=forms.Textarea)


# Create form with invalid data (content too short)
form = CommentCreateForm(data={'content': 'ok'})

# >>> form.is_valid()
# >>> form.errors
# >>> form.errors.get('content', [])
# >>> form.non_field_errors()

<br>

## Field-Level Validation

---

### `clean_fieldname()` Methods

---

For custom validation on a single field, define a method named `clean_<fieldname>()`:

In [ ]:
# blog/forms.py — clean_<fieldname>() on BlogModelForm.author
from django import forms
from django.core.exceptions import ValidationError
from blog.models import Blog, CategoryType


class BlogModelForm(forms.ModelForm):
    """ModelForm for creating a Blog with field-level custom validation."""

    class Meta:
        model = Blog
        fields = ['title', 'author', 'author_email', 'published_date', 'category_type']

    def clean_author(self):
        # value has already passed the basic field validation
        author = self.cleaned_data['author']  

        # Disallow reserved/system account names
        forbidden = ['admin', 'root', 'anonymous']
        if author.lower() in forbidden:
            raise ValidationError(f"The name '{author}' is reserved and not allowed.")

        # Always return the value — here we also normalise it to lowercase
        return author.lower()

In [ ]:
# blog/views.py — BlogModelForm with clean_author()
from django.views.decorators.csrf import csrf_exempt
from django.http import JsonResponse
import json

from blog.models import Blog

@csrf_exempt  # Only for demo/curl testing - in production use CSRF tokens
def blog_create(request):
    if request.method != 'POST':
        return JsonResponse({'tip': 'Send POST with title, author, author_email, published_date, category_type'})

    data = json.loads(request.body)
    form = BlogModelForm(data)  # uses clean_author() for extra validation
    if not form.is_valid():
        return JsonResponse({'errors': form.errors}, status=400)

    blog = form.save()  # ModelForm — save() creates the Blog instance
    return JsonResponse({'id': blog.id, 'title': blog.title}, status=201)

In [ ]:
# SHELL-EXAMPLE: test clean_author() on BlogModelForm
from blog.forms import BlogModelForm

# Forbidden author name
form1 = BlogModelForm(data={
    'title': 'My Post',
    'author': 'Admin',
    'published_date': '2026-04-13',
    'category_type': 1,
})
# >>> form1.is_valid()
# >>> form1.errors

# Valid data — author gets lowercased by clean_author()
form2 = BlogModelForm(data={
    'title': 'My Post',
    'author': 'matous_holinka',
    'published_date': '2026-04-13',
    'category_type': 1,
})
print(f"Valid: {form2.is_valid()}")
if form2.is_valid():
    print(f"Cleaned author: {form2.cleaned_data['author']}")


# happy-case
curl -s -X POST http://127.0.0.1:8000/blog/api/blogs/ \
  -H "Content-Type: application/json" \
  -d '{"title":"My Post","author":"matoush","published_date":"2026-04-14","category_type":1}' | python3 -m json.tool

# error-case
curl -s -X POST http://127.0.0.1:8000/blog/api/blogs/ \
  -H "Content-Type: application/json" \
  -d '{"title":"My Post","author":"admin","published_date":"2026-04-14","category_type":1}' | python3 -m json.tool


<br>

**Rules for `clean_<fieldname>()` methods:**

| Rule | Description |
|------|-------------|
| Access via `self.cleaned_data` | The value is already type-converted |
| Raise `ValidationError` | For invalid values |
| Return the value | Always return the (possibly modified) value |
| Return value goes to `cleaned_data` | The returned value replaces the original |

<br>

### Accessing Other Field Values

---

In `clean_<fieldname>()`, you can access other fields that have already been cleaned:

In [ ]:
# blog/forms.py — clean_<fieldname>() accessing another field
# Rule: BlogReview.comment is required when rating <= 2 (negative review needs explanation)
from django import forms
from django.core.exceptions import ValidationError


class BlogReviewForm(forms.Form):
    """BlogReview form with cross-field check inside clean_comment()."""
    blog = forms.ModelChoiceField(
        queryset=Blog.objects.all(),
        label='Blog post',
    )
    rating = forms.IntegerField(min_value=1, max_value=5)
    comment = forms.CharField(required=False, widget=forms.Textarea)

    def clean_comment(self):
        """Comment is mandatory for low ratings (1–2 stars)."""
        comment = self.cleaned_data.get('comment', '')

        # rating may be absent if it failed its own validation
        rating = self.cleaned_data.get('rating')

        if rating is not None and rating <= 2 and not comment:
            raise ValidationError(
                "Please explain your low rating — comment is required for ratings of 1 or 2."
            )

        return comment

In [ ]:
# Test clean_comment() on BlogReviewForm

# Low rating without comment → should fail
form1 = BlogReviewForm(data={
    'blog': 1,       # ID of an existing Blog
    'rating': 2,
    'comment': '',
})
print(f"Low rating, no comment valid: {form1.is_valid()}")
print(f"Comment errors: {form1.errors.get('comment', [])}")

print()

# Low rating with comment → should pass
form2 = BlogReviewForm(data={
    'blog': 1,
    'rating': 1,
    'comment': 'The article was poorly structured and contained factual errors.',
})
print(f"Low rating with comment valid: {form2.is_valid()}")
print(f"Errors: {form2.errors}")

print()

# High rating without comment → should pass (comment only required for 1–2 stars)
form3 = BlogReviewForm(data={
    'blog': 1,
    'rating': 4,
    'comment': '',
})
print(f"High rating, no comment valid: {form3.is_valid()}")
print(f"Errors: {form3.errors}")

<br>

**Caution:** Fields are cleaned in declaration order. When accessing `self.cleaned_data['other_field']`, the other field must be declared **before** the current field in the form class.

<br>

## Form-Level Validation

---

### The `clean()` Method

---

**The Collective Clean:** Use the `clean()` method when validation depends on multiple fields working together (Cross-field validation).

**When to use it:** For logic like `'Password Confirmation'` (comparing two fields) or `'Minimum Order Value'` (checking a product selection against a total price).

<br>

For validation that involves multiple fields together, override the `clean()` method:

In [ ]:
# blog/forms.py — clean() for cross-field validation on BlogReviewForm
from django import forms
from django.core.exceptions import ValidationError
from blog.models import Blog


class BlogReviewForm(forms.Form):
    blog = forms.ModelChoiceField(queryset=Blog.objects.all())
    rating = forms.IntegerField(min_value=1, max_value=5)
    comment = forms.CharField(required=False, widget=forms.Textarea)

    def clean(self):
        """Validate that a 5-star or 1-star review always includes a comment."""

        cleaned_data = super().clean()

        rating = cleaned_data.get('rating')
        comment = cleaned_data.get('comment')

        # Both values must have passed field-level validation first
        if rating == 5 and not comment:
            raise ValidationError(
                "A 5-star review should include a comment explaining what was great."
            )

        if rating == 1 and not comment:
            raise ValidationError(
                "A 1-star review must include a comment explaining the issue."
            )

        return cleaned_data

In [ ]:
# Test clean() on BlogReviewForm
from blog.forms import BlogReviewForm

# 5-star with no comment → should fail
form = BlogReviewForm(data={
    'blog': 1,
    'rating': 5,
    'comment': '',
})

# >>> form.is_valid()
# >>> form.errors
# >>> form.non_field_errors()

# 5-star with comment → should pass
form2 = BlogReviewForm(data={
    'blog': 1,
    'rating': 5,
    'comment': 'Excellent deep-dive into Django forms!',
})

# >>> form2.is_valid()
# >>> form2.errors
# >>> form2.non_field_errors()

# Note: review endpoint (/blog/review-form/) is an HTML view with @login_required,
# not a JSON API — test via browser or Python shell, not curl.

if the error and its original field are too vague, use `self.add_error()` method.
```python
self.add_error(
    'comment',
    "A 5-star review should include a comment explaining what was great."
)
```

<br>

### Cross-Field Validation

---

A more complex example with date range validation:

In [ ]:
# blog/forms.py — clean() with date cross-field validation on BlogModelForm
from django import forms
from django.core.exceptions import ValidationError
from datetime import date
from blog.models import Blog


class BlogModelForm(forms.ModelForm):
    """Blog ModelForm: published_date must not be more than 1 year in the future."""

    class Meta:
        model = Blog
        fields = ['title', 'author', 'published_date', 'category_type']
        widgets = {
            'published_date': forms.DateInput(attrs={'type': 'date'}),
        }

    def clean(self):
        """Validate date logic across fields."""
        cleaned_data = super().clean()
        published_date = cleaned_data.get('published_date')

        if published_date:
            today = date.today()

            days_ahead = (published_date - today).days
            if days_ahead > 1:
                raise ValidationError(
                    f"Published date is {days_ahead} days in the future — "
                    "Published date cannot be in the future."
                )

            days_old = (date.today() - published_date).days
            if days_old > 365:
                raise ValidationError(
                    f"Published date is {days_old} days in the past — cannot be older than 1 year."
                )

        return cleaned_data

# Error-case
curl -s -X POST http://127.0.0.1:8000/blog/api/blogs/ \
  -H "Content-Type: application/json" \
  -d '{"title":"Old Post","author":"matous h","published_date":"2024-01-01","category_type":1}' | python3 -m json.tool


# Happy-case
curl -s -X POST http://127.0.0.1:8000/blog/api/blogs/ \
  -H "Content-Type: application/json" \
  -d '{"title":"Recent Post","author":"matoush","published_date":"2026-01-01","category_type":1}' | python3 -m json.tool

<br>

**Note the difference:**

| Method | Error Type |
|--------|------------|
| `raise ValidationError(...)` | Creates a non-field error |
| `self.add_error('fieldname', ...)` | Adds error to specific field |

<br>

## Custom Validators

---

### Creating Reusable Validators

---

For validation logic you want to reuse across multiple forms or fields, create standalone validator functions:

In [ ]:
# blog/validators.py — reusable validators for Blog / Comment

from django.core.exceptions import ValidationError
import re


def validate_only_letters_and_spaces(value: str) -> None:
    """Allow only letters and spaces (used for Blog.author)."""

    if not re.match(r'^[a-zA-Z ]+$', value):
        raise ValidationError(
            "Only letters and spaces are allowed.",
            code='invalid_characters',
        )


def validate_no_spam_words(value: str) -> None:
    """Reject comment content that looks like spam (used for Comment.content)."""

    spam_words = ['buy now', 'click here', 'free money']

    for word in spam_words:
        if word in value.lower():
            raise ValidationError(
                "This content looks like spam and cannot be submitted.",
                code='spam',
            )

In [ ]:
# blog/forms.py — BlogModelForm and CommentCreateForm using custom validators
from django import forms
from blog.models import Blog, CategoryType
from blog.validators import validate_only_letters_and_spaces, validate_no_spam_words


class BlogModelForm(forms.ModelForm):
    """BlogModelForm using validate_only_letters_and_spaces for author."""

    class Meta:
        model = Blog
        fields = ['title', 'author', 'author_email', 'published_date', 'category_type']

    # Override the author field to add a custom validator
    author = forms.CharField(
        max_length=100,
        validators=[validate_only_letters_and_spaces],  # reusable validator
    )


class CommentCreateForm(forms.Form):
    """CommentCreateForm using validate_no_spam_words for content."""

    content = forms.CharField(
        min_length=3,
        widget=forms.Textarea,
        validators=[validate_no_spam_words],  # reusable validator
    )

In [ ]:
# Test custom validators on project forms
from blog.forms import BlogModelForm, CommentCreateForm

# Spam comment content
form1 = CommentCreateForm(data={'content': \
    'click here to win free money!'})
# >>> form1.is_valid()
# >>> form1.errors
# >>> form1.errors.get('content', [])

form2 = CommentCreateForm(data={'content': 'Really helpful blog , thank you!'})
# >>> form2.is_valid()
# >>> form2.errors
# >>> form2.errors.get('author', [])

In [ ]:
# blog/views.py
# ...

@csrf_exempt
@require_http_methods(["POST"])
def api_comment_create(request: HttpRequest) -> JsonResponse:
    data = json.loads(request.body)
    form = CommentCreateForm(data)
    if not form.is_valid():
        return JsonResponse({'errors': form.errors}, status=400)
    return JsonResponse({'status': 'ok', 'data': form.cleaned_data}, status=201)

In [ ]:
# blog/urls.py
# ...

urlpatterns = [
    # ...
    path('api/blogs/', views.api_blog_list_create, name='api_blog_list_create'),
    path('api/comments/', views.api_comment_create, name='api_comment_create'),
]

# happy-case
curl -s -X POST http://127.0.0.1:8000/blog/api/comments/ \
  -H "Content-Type: application/json" \
  -d '{"content":"This article was very helpful, thank you."}' | python3 -m json.tool

# error-case
curl -s -X POST http://127.0.0.1:8000/blog/api/comments/ \
  -H "Content-Type: application/json" \
  -d '{"content":"Click here to win free money now!"}' | python3 -m json.tool

<br>

### Validators with Parameters

---

For validators that need configuration, create a class:

In [ ]:
# blog/validators.py — parameterised validators for Blog and Comment

from django.core.exceptions import ValidationError
from django.utils.deconstruct import deconstructible


@deconstructible  # required if used in model field validators (migrations)
class MinWordCountValidator:
    """Validate that a text field contains at least min_words words.

    Used for:
      - Blog.content  (long-form body, should be substantial)
      - Comment.content (should be more than one word)
    """

    def __init__(self, min_words: int):
        self.min_words = min_words

    def __call__(self, value: str):
        word_count = len(value.split())
        if word_count < self.min_words:
            raise ValidationError(
                f"Please write at least {self.min_words} words. "
                f"Current count: {word_count}."
            )

    def __eq__(self, other):
        return (
            isinstance(other, MinWordCountValidator)
            and self.min_words == other.min_words
        )

**Class-Based Validators breakdown:**

`__init__` (Setup): Used to parameterize the validator (e.g., setting the minimum word count).

`__call__` (The Logic): This makes the class instance 'callable' like a function. It's where the actual validation check happens.

`@deconstructible` & `__eq__` (Serialization): Necessary for Django Migrations. They allow Django to save the validator's state into the database schema and compare different validator instances.

In [ ]:
# blog/forms.py — MinWordCountValidator applied to Blog and Comment forms
from django import forms
from blog.models import Blog, CategoryType
# from blog.validators import MinWordCountValidator


class BlogModelForm(forms.ModelForm):
    """Blog post body must contain at least 50 words."""

    class Meta:
        model = Blog
        fields = ['title', 'author', 'published_date', 'category_type', 'content']

    # Override content field to add MinWordCountValidator
    content = forms.CharField(
        widget=forms.Textarea,
        validators=[MinWordCountValidator(min_words=50)],  # Blog body must be substantial
    )


class CommentCreateForm(forms.Form):
    """Comment must be at least 5 words (not just 'ok' or 'nice')."""

    content = forms.CharField(
        min_length=3,
        widget=forms.Textarea,
        validators=[MinWordCountValidator(min_words=5)],
    )

In [ ]:
# Test MinWordCountValidator on BlogModelForm and CommentCreateForm
from blog.forms import CommentCreateForm

# Blog content too short
form1 = BlogModelForm(data={
    'title': 'Django Tips',
    'author': 'matous_holinka',
    'published_date': '2026-04-13',
    'category_type': 1,
    'content': 'Too short.',  # less than 50 words
})
print(f"Short blog content valid: {form1.is_valid()}")
print(f"Content errors: {form1.errors.get('content', [])}")

print()

# Comment too short (below 5 words)
form2 = CommentCreateForm(data={'content': 'Nice post.'})
print(f"Short comment valid: {form2.is_valid()}")
print(f"Content errors: {form2.errors.get('content', [])}")

<br>

## Error Handling and Display

---

### Adding Custom Errors

---

Use `add_error()` to add errors to specific fields or the form as a whole:

In [ ]:
# blog/forms.py — add_error() on BlogReviewForm
from django import forms
from django.core.exceptions import ValidationError
from blog.models import BlogReview


class BlogReviewForm(forms.Form):
    """BlogReview form: uses add_error() to attach errors to specific fields."""

    blog_id = forms.IntegerField()
    rating = forms.IntegerField(min_value=1, max_value=5)
    comment = forms.CharField(required=False, widget=forms.Textarea)

    def clean(self):
        cleaned_data = super().clean()

        blog_id = cleaned_data.get('blog_id')
        rating = cleaned_data.get('rating')
        comment = cleaned_data.get('comment', '')

        # Low rating must include a comment (attach error to the comment field)
        if rating is not None and rating <= 2 and not comment:
            self.add_error(
                'comment',
                "A comment is required when rating is 1 or 2 stars.",
            )

        # Prevent duplicate review for the same blog (simulated DB check)
        # In a real view you'd pass request.user and check BlogReview.objects
        existing_ids = []  # e.g. BlogReview.objects.filter(user=request.user).values_list('blog_id', flat=True)
        if blog_id in existing_ids:
            self.add_error(
                'blog_id',
                "You have already reviewed this blog post.",
            )

        return cleaned_data

<br>

### Non-Field Errors

---

For errors that aren't specific to any field, use `add_error(None, ...)` or raise `ValidationError` in `clean()`:

In [ ]:
# blog/forms.py — non-field errors on BlogReviewForm
from django import forms
from django.core.exceptions import ValidationError


class BlogReviewForm(forms.Form):
    """BlogReview form: non-field error when both rating and comment are empty."""

    rating = forms.IntegerField(min_value=1, max_value=5)
    comment = forms.CharField(required=False, widget=forms.Textarea)

    def clean(self):
        cleaned_data = super().clean()

        rating = cleaned_data.get('rating')
        comment = cleaned_data.get('comment', '')

        # Business rule: 5-star with no comment OR 1-star with no comment
        # → non-field error because it involves both fields equally
        if rating in (1, 5) and not comment:
            self.add_error(
                # Its not a good idea to skip the field name
                None,  # None → non-field error
                f"A {'5-star' if rating == 5 else '1-star'} review must include a written comment.",
            )

        # Alternative: raise raises a non-field error directly
        # raise ValidationError("Please fill in at least one field.")

        return cleaned_data

In [ ]:
# Test non-field errors on BlogReviewForm

# 1-star with no comment → non-field error
form = BlogReviewForm(data={
    'rating': 1,
    'comment': '',
})

print(f"Valid: {form.is_valid()}")
print(f"Non-field errors: {form.non_field_errors()}")
print(f"Field errors: {form.errors}")

print()

# 5-star with comment → valid
form2 = BlogReviewForm(data={
    'rating': 5,
    'comment': 'Best Django tutorial I have read this year!',
})
print(f"5-star with comment valid: {form2.is_valid()}")

<br>

### Displaying Errors in Templates

---

Here's how to display all types of errors in your templates (server-side):

In [ ]:
# templates/blog/comment_form.html — complete error-handling template

template_with_errors = """
{% extends 'blog/base.html' %}

{% block title %}Submit Review{% endblock %}

{% block content %}
<h1>Submit a Blog Review</h1>

<form method="post" novalidate>
    {% csrf_token %}

    {# Non-field errors (e.g. "1-star review must include a comment") #}
    {% if form.non_field_errors %}
        <div class="alert alert-danger">
            <strong>Please correct the following:</strong>
            <ul>
                {% for error in form.non_field_errors %}
                    <li>{{ error }}</li>
                {% endfor %}
            </ul>
        </div>
    {% endif %}

    {# Render each field with its label, widget, help text, and errors #}
    {% for field in form %}
        <div class="form-group {% if field.errors %}has-error{% endif %}">
            <label for="{{ field.id_for_label }}">
                {{ field.label }}
                {% if field.field.required %}<span class="required">*</span>{% endif %}
            </label>

            {{ field }}

            {% if field.help_text %}
                <small class="help-text">{{ field.help_text }}</small>
            {% endif %}

            {% if field.errors %}
                <ul class="error-list">
                    {% for error in field.errors %}
                        <li class="error">{{ error }}</li>
                    {% endfor %}
                </ul>
            {% endif %}
        </div>
    {% endfor %}

    <button type="submit">Submit Review</button>
</form>
{% endblock %}
"""
print(template_with_errors)

<br>

## Summary

---

| Concept | Key Points |
|---------|------------|
| **Validation flow** | to_python → validate → validators → clean_field → clean |
| **clean_fieldname()** | Custom validation for single field, return cleaned value |
| **clean()** | Cross-field validation, call super().clean() first |
| **ValidationError** | Raise to indicate invalid data, include helpful message |
| **add_error()** | Add error to specific field or None for non-field |
| **Custom validators** | Reusable functions/classes, use @deconstructible for models |
| **form.errors** | Dictionary of all field errors |
| **form.non_field_errors()** | List of errors not tied to any field |

<br>

### 🧠 EXERCISE 🧠, Password Validation Form

---

Create a password change form with comprehensive validation:

1. Create a `PasswordForm` with fields:
   - `current_password`
   - `new_password`
   - `confirm_password`

2. Add field-level validation for `new_password`:
   - At least 8 characters
   - Must contain at least one uppercase letter
   - Must contain at least one digit
   - Must contain at least one special character (!@#$%^&*)

3. Add form-level validation:
   - `new_password` and `confirm_password` must match
   - `new_password` must be different from `current_password`

4. Test with various inputs

<details>
    <summary>▶️ Solution</summary>
    
```python
import re
from django import forms
from django.core.exceptions import ValidationError


class PasswordForm(forms.Form):
    """Password change form with comprehensive validation."""
    
    current_password = forms.CharField(
        widget=forms.PasswordInput,
        label='Current Password'
    )
    
    new_password = forms.CharField(
        widget=forms.PasswordInput,
        label='New Password',
        help_text='At least 8 characters with uppercase, digit, and special character.'
    )
    
    confirm_password = forms.CharField(
        widget=forms.PasswordInput,
        label='Confirm New Password'
    )
    
    def clean_new_password(self):
        """Validate password strength."""
        
        password = self.cleaned_data['new_password']
        
        # Check length
        if len(password) < 8:
            raise ValidationError(
                "Password must be at least 8 characters long."
            )
        
        # Check for uppercase
        if not re.search(r'[A-Z]', password):
            raise ValidationError(
                "Password must contain at least one uppercase letter."
            )
        
        # Check for digit
        if not re.search(r'\d', password):
            raise ValidationError(
                "Password must contain at least one digit."
            )
        
        # Check for special character
        if not re.search(r'[!@#$%^&*]', password):
            raise ValidationError(
                "Password must contain at least one special character (!@#$%^&*)."
            )
        
        return password
    
    def clean(self):
        """Cross-field validation."""
        
        cleaned_data = super().clean()
        
        current = cleaned_data.get('current_password')
        new = cleaned_data.get('new_password')
        confirm = cleaned_data.get('confirm_password')
        
        # Check passwords match
        if new and confirm and new != confirm:
            raise ValidationError(
                "The two password fields must match."
            )
        
        # Check new is different from current
        if current and new and current == new:
            self.add_error(
                'new_password',
                "New password must be different from current password."
            )
        
        return cleaned_data


# Test the form
test_cases = [
    # Too short
    {'current_password': 'old', 'new_password': 'Short1!', 'confirm_password': 'Short1!'},
    # No uppercase
    {'current_password': 'old', 'new_password': 'password1!', 'confirm_password': 'password1!'},
    # No digit
    {'current_password': 'old', 'new_password': 'Password!', 'confirm_password': 'Password!'},
    # No special char
    {'current_password': 'old', 'new_password': 'Password1', 'confirm_password': 'Password1'},
    # Don't match
    {'current_password': 'old', 'new_password': 'Password1!', 'confirm_password': 'Different1!'},
    # Valid
    {'current_password': 'old', 'new_password': 'NewPass1!', 'confirm_password': 'NewPass1!'},
]

for data in test_cases:
    form = PasswordForm(data=data)
    print(f"Valid: {form.is_valid()}, Errors: {form.errors}")
```
</details>

<br>

### 🧠 EXERCISE 🧠, Date Range Validation

---

Create a booking form with date validation:

1. Create a `BookingForm` with:
   - `guest_name` (CharField)
   - `check_in` (DateField)
   - `check_out` (DateField)
   - `room_type` (ChoiceField: 'standard', 'deluxe', 'suite')

2. Validate:
   - Check-in must be today or in the future
   - Check-out must be after check-in
   - Maximum stay is 30 days
   - Suite bookings require at least 2 nights

3. Display appropriate error messages

<details>
    <summary>▶️ Solution</summary>
    
```python
from django import forms
from django.core.exceptions import ValidationError
from datetime import date


class BookingForm(forms.Form):
    """Hotel booking form with date validation."""
    
    ROOM_CHOICES = [
        ('standard', 'Standard Room'),
        ('deluxe', 'Deluxe Room'),
        ('suite', 'Suite'),
    ]
    
    guest_name = forms.CharField(
        max_length=100,
        label='Guest Name'
    )
    
    check_in = forms.DateField(
        widget=forms.DateInput(attrs={'type': 'date'}),
        label='Check-in Date'
    )
    
    check_out = forms.DateField(
        widget=forms.DateInput(attrs={'type': 'date'}),
        label='Check-out Date'
    )
    
    room_type = forms.ChoiceField(
        choices=ROOM_CHOICES,
        label='Room Type'
    )
    
    def clean_check_in(self):
        """Validate check-in is not in the past."""
        
        check_in = self.cleaned_data['check_in']
        
        if check_in < date.today():
            raise ValidationError(
                "Check-in date cannot be in the past."
            )
        
        return check_in
    
    def clean(self):
        """Cross-field validation for dates and room type."""
        
        cleaned_data = super().clean()
        
        check_in = cleaned_data.get('check_in')
        check_out = cleaned_data.get('check_out')
        room_type = cleaned_data.get('room_type')
        
        if check_in and check_out:
            # Check-out must be after check-in
            if check_out <= check_in:
                self.add_error(
                    'check_out',
                    "Check-out must be after check-in date."
                )
                return cleaned_data
            
            # Calculate duration
            duration = (check_out - check_in).days
            
            # Maximum 30 days
            if duration > 30:
                raise ValidationError(
                    f"Maximum stay is 30 days. Selected duration: {duration} days."
                )
            
            # Suite requires minimum 2 nights
            if room_type == 'suite' and duration < 2:
                self.add_error(
                    'room_type',
                    "Suite bookings require a minimum stay of 2 nights."
                )
        
        return cleaned_data


# Test cases
from datetime import timedelta

today = date.today()

# Valid booking
form1 = BookingForm(data={
    'guest_name': 'John Doe',
    'check_in': today + timedelta(days=1),
    'check_out': today + timedelta(days=5),
    'room_type': 'standard'
})
print(f"Valid booking: {form1.is_valid()}")

# Suite with 1 night
form2 = BookingForm(data={
    'guest_name': 'Jane Doe',
    'check_in': today + timedelta(days=1),
    'check_out': today + timedelta(days=2),
    'room_type': 'suite'
})
print(f"Suite 1 night valid: {form2.is_valid()}")
print(f"Suite errors: {form2.errors}")
```
</details>

---